In [ ]:
import sys
# sys.path.insert(0, '../../Utilities/')

# from jax.example_libraries import optimizers
import jax.numpy as jnp
import numpy as np
from jax import grad, jit, vmap, value_and_grad, hessian
from jax import random
from functools import partial
import flax.linen as nn
import jax
import optax
import pickle
from datetime import datetime
import os
import numpy as np
from scipy import sparse
import time
from jax import grad, hessian, jit, vmap, pmap
from jax.tree_util import tree_leaves
from datetime import datetime
import matplotlib.pyplot as plt
import pickle
import flax.linen as nn
import optax
import optax.tree_utils as otu
from NN_class import MLP
from my_helpers import plot_u_theta, plot_kh, plot_loss_history
from functools import partial
from Pwp_class import PiecewiseLinearPolynomials
from minmax_opt_class import pytree_l2_norm, MinMaxOptProb
from seq_minmax_opt_class import SequentialMinMaxOptProb


import matplotlib.pyplot as plt
import scipy.io
from scipy.interpolate import griddata

import time
import matplotlib.gridspec as gridspec
from mpl_toolkits.axes_grid1 import make_axes_locatable
import mpmath

# Generate key which is used to generate random numbers
key = random.PRNGKey(1)

In [ ]:
def u0(x):
    u = jnp.zeros_like(x)
    u = jnp.where((x > 0.0) & (x < 1.0), 1.0, u)
    return u

def u_exact(t, x):
    # Initialize u as zero
    u = 0.0
    # Condition 1: x < 0, u = 0 (already zero by default)
    # Condition 2: 0 < x < t and t > 0, u = x/t
    u = jnp.where((x > 0) & (x < t) & (t > 0), x / t, u)
    
    # Condition 3: t < x < 1 + t/2, u = 1
    u = jnp.where((x > t) & (x < 1 + t / 2), 1.0, u)
    
    # Condition 4: x > 1 + t/2, u = 0 (already zero by default)
    return u
vmap_u_exact = vmap(u_exact, in_axes=(0, 0), out_axes=0)

# Boundary at x=-1
def u1(x):
    return 0
vmap_u1 = vmap(u1, in_axes=0, out_axes=0)


# Boundary at x=2
def u2(x):
    return 0
vmap_u2 = vmap(u2, in_axes=0, out_axes=0)

def res_visc(u_quad, u_t, u_x, u_xx, nu):
    return u_t + u_quad * u_x - nu*u_xx
def res(u_quad, u_t, u_x):
    return u_t + u_quad * u_x


t_min = 0
t_max = 2.5
num_segments = 5
x_min = -0.5
x_max = 2.5
Nt = 101
Nx = 401
layers = [2, 50, 50, 50, 50, 50, 1]
hidden_dims = layers[1:-1]

# Domain bounds
lb = jnp.array([t_min, x_min])
ub = jnp.array([t_max, x_max])

In [ ]:
seq_trainer = SequentialMinMaxOptProb(
    num_segments=num_segments,
    total_t_min=t_min,
    total_t_max=t_max,
    seq_T = [0,1,2,2.5],
    N_t=16,
    N_x=48,
    lr_theta=0.0002,
    hidden_dims=hidden_dims,
    seed=0,
    N_bc=Nt,
    N_ic=Nx,
    x_min=x_min, x_max=x_max,
    u0=u0, u1=vmap_u1, u2=vmap_u2,
    nu=0.01,
    res_visc=res_visc,
    res=res,
)


In [ ]:
seq_trainer.train_sequential(train_num_iterations=0, pre_train_num_iterations=10000)

In [ ]:
plot_combined_u_theta(seq_trainer)

In [ ]:
seq_trainer_us = SequentialMinMaxOptProb(
    num_segments=num_segments,
    total_t_min=t_min,
    total_t_max=t_max,
    seq_T = [0, 0.5, 1, 1.5, 2, 2.5],
    N_t=16,
    N_x=48,
    lr_theta=0.0002,
    hidden_dims=hidden_dims,
    seed=0,
    N_bc=Nt,
    N_ic=Nx,
    x_min=x_min, x_max=x_max,
    u0=u0, u1=vmap_u1, u2=vmap_u2,
    nu=0.01,
    res_visc=res_visc,
    res=res,
)

seq_trainer_us.train_sequential(train_num_iterations=5000, pre_train_num_iterations=0)

In [ ]:
plot_combined_u_theta(seq_trainer)

In [ ]:
seq_trainer_us.segments[0].pwp.N_x


In [ ]:
seq_trainer_us.seq_T = jnp.array(seq_trainer_us.seq_T)
seq_trainer_us.plot_combined_u_theta(500, M_x=500)

In [ ]:
seq_trainer_us = SequentialMinMaxOptProb(
    num_segments=num_segments,
    total_t_min=t_min,
    total_t_max=t_max,
    seq_T = [0, 0.5, 1, 1.5, 2, 2.5],
    N_t=16,
    N_x=48,
    lr_theta=0.0002,
    hidden_dims=hidden_dims,
    seed=0,
    N_bc=Nt,
    N_ic=Nx,
    x_min=x_min, x_max=x_max,
    u0=u0, u1=vmap_u1, u2=vmap_u2,
    nu=0.01,
    res_visc=res_visc,
    res=res,
)

seq_trainer_us.train_sequential(train_num_iterations=5000, pre_train_num_iterations=0)

In [ ]:
seq_trainer_us = SequentialMinMaxOptProb(
    num_segments=num_segments,
    total_t_min=t_min,
    total_t_max=t_max,
    seq_T = np.linspace(0, 2.5, 26),
    N_t=16,
    N_x=48,
    lr_theta=0.0002,
    hidden_dims=hidden_dims,
    seed=0,
    N_bc=Nt,
    N_ic=Nx,
    x_min=x_min, x_max=x_max,
    u0=u0, u1=vmap_u1, u2=vmap_u2,
    nu=0.01,
    res_visc=res_visc,
    res=res,
)

seq_trainer_us.train_sequential(train_num_iterations=2000, pre_train_num_iterations=0)

In [ ]:
seq_trainer_us.plot_combined_u_theta(500, M_x=500)

In [ ]:
def plot_solution_at_time(self, T, M_x=100):
    """
    Plot the solution u_theta at a specific time T.

    Args:
        T (float): The time at which to plot the solution.
        M_x (int): Number of grid points in the x-direction (default: 100).
    """
    # Find the segment that contains time T
    segment_index = None
    for i in range(len(self.seq_T) - 1):
        if self.seq_T[i] <= T < self.seq_T[i + 1]:
            segment_index = i
            break
    if segment_index is None:
        if T == self.seq_T[-1]:  # Allow T to be exactly at the end
            segment_index = len(self.seq_T) - 2
        else:
            raise ValueError(f"Time T={T} is outside the defined segments [{self.seq_T[0]}, {self.seq_T[-1]}]")

    # Get the segment
    segment = self.segments[segment_index]

    # Generate spatial grid and input points
    x_grid = jnp.linspace(self.x_min, self.x_max, M_x)
    t_fixed = jnp.full_like(x_grid, T)  # Array of T repeated for each x
    tx_points = jnp.stack([t_fixed, x_grid], axis=1)  # Shape: (M_x, 2)

    # Evaluate the neural network at these points
    u_values = segment.model(segment.params, tx_points)  # Shape: (M_x,)

    # Plot the solution
    plt.figure(figsize=(8, 6))
    plt.plot(x_grid, u_values, label=f'u_theta at t={T}')
    plt.xlabel('x')
    plt.ylabel('u_theta(T, x)')
    plt.title(f'Solution at Time T={T}')
    plt.legend()
    plt.show()

In [ ]:
plot_solution_at_time(seq_trainer_us, 2)